#  Validation Summary

### 1. Demand Growth
- Baseline Total Consumption: `3,984,017 kWh`
- Projected Total Consumption: `4,349,768 kWh`
- Synthetic Growth: **9.18%**
- EPRA Benchmark Growth: **8.25%**

### 2. Regional Alignment
- Mean Absolute Error vs EPRA proportions ≈ **0.002**
- Dataset aligned with official regional shares.

### 3. Regional Shocks
- South Nyanza: **+22.53%**
- North Eastern: **+13.73%**
- Other regions: ~**8.25%**

### 4. Data Integrity
- No missing values.
- Negative consumption values clipped to zero for realism.

### 5. Policy Anchors
- Grid Renewable Share: **78.79%** (H2 2025)
- Installed Capacity Share: **80.6%**
- REP Levy: **5%** revenue premium included.


In [18]:
import pandas as pd

synthetic_dataset = pd.read_csv("synthetic_dataset.csv")
synthetic_dataset.head(9994)


,household_id,region,cooking_fuel,consumption_kwh,consumption_kwh_projected
0,1,Nairobi,charcoal,416.114069,450.443479
1,2,Nairobi,charcoal,788.856868,853.937560
2,3,Nairobi,LPG,265.797006,287.725259
3,4,Nairobi,charcoal,545.066227,590.034191
4,5,Nairobi,firewood,372.099827,402.798063
...,...,...,...,...,...
9989,9990,South Nyanza,firewood,781.119605,957.105852
9990,9991,South Nyanza,LPG,665.268265,815.153206
9991,9992,South Nyanza,charcoal,296.407049,363.187557
9992,9993,South Nyanza,LPG,543.691410,666.185084


In [9]:
synthetic_dataset.info()
synthetic_dataset.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   household_id               10000 non-null  int64  
 1   region                     10000 non-null  object 
 2   cooking_fuel               10000 non-null  object 
 3   consumption_kwh            10000 non-null  float64
 4   consumption_kwh_projected  10000 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 390.8+ KB


household_id                 0
region                       0
cooking_fuel                 0
consumption_kwh              0
consumption_kwh_projected    0
dtype: int64

In [10]:
regional_totals = synthetic_dataset.groupby("region")["consumption_kwh"].sum()
regional_proportions = regional_totals / regional_totals.sum()
print(regional_proportions)


region
Central Rift     0.090940
Coast            0.168912
Mt. Kenya        0.066087
Nairobi          0.437116
North Eastern    0.113588
North Rift       0.048757
South Nyanza     0.021568
West Kenya       0.053031
Name: consumption_kwh, dtype: float64


In [11]:
baseline_total = synthetic_dataset["consumption_kwh"].sum()
projected_total = synthetic_dataset["consumption_kwh_projected"].sum()
synthetic_growth = (projected_total / baseline_total) - 1
print("Synthetic Growth:", round(synthetic_growth, 4))


Synthetic Growth: 0.0918


In [12]:
regional_projection = synthetic_dataset.groupby("region")[["consumption_kwh","consumption_kwh_projected"]].sum()
regional_projection["growth_rate"] = (regional_projection["consumption_kwh_projected"] / regional_projection["consumption_kwh"]) - 1
print(regional_projection)


               consumption_kwh  consumption_kwh_projected  growth_rate
region                                                                
Central Rift      3.623081e+05               3.921985e+05       0.0825
Coast             6.729501e+05               7.284685e+05       0.0825
Mt. Kenya         2.632933e+05               2.850150e+05       0.0825
Nairobi           1.741477e+06               1.885148e+06       0.0825
North Eastern     4.525377e+05               5.146711e+05       0.1373
North Rift        1.942498e+05               2.102754e+05       0.0825
South Nyanza      8.592569e+04               1.052847e+05       0.2253
West Kenya        2.112766e+05               2.287069e+05       0.0825


In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("synthetic_dataset.csv")
features = df[["consumption_kwh_projected", "income_proxy", "appliance_index"]]

scaler = StandardScaler()
X = scaler.fit_transform(features)


KeyError: "['income_proxy', 'appliance_index'] not in index"

In [4]:
import pandas as pd

df = pd.read_csv("synthetic_dataset.csv")

# Example income proxy from cooking fuel
fuel_income_map = {
    "firewood": 1, "charcoal": 2, "kerosene": 3,
    "LPG": 4, "electric": 5
}
df["income_proxy"] = df["cooking_fuel"].map(fuel_income_map)

# Appliance index (if appliance columns exist)
appliance_cols = [c for c in df.columns if c in ["tv","fridge","radio","washing_machine"]]
df["appliance_index"] = df[appliance_cols].sum(axis=1) if appliance_cols else 0


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

features = df[["consumption_kwh", "income_proxy", "appliance_index"]]
X = StandardScaler().fit_transform(features)

# K-Means
kmeans = KMeans(n_clusters=4, random_state=42)
df["segment_kmeans"] = kmeans.fit_predict(X)

# GMM
gmm = GaussianMixture(n_components=4, random_state=42)
df["segment_gmm"] = gmm.fit_predict(X)

print("KMeans Silhouette:", silhouette_score(X, df["segment_kmeans"]))
print("GMM Silhouette:", silhouette_score(X, df["segment_gmm"]))


ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [6]:
print(df[["consumption_kwh", "income_proxy", "appliance_index"]].isna().sum())


consumption_kwh       0
income_proxy       2580
appliance_index       0
dtype: int64


In [7]:
fuel_income_map = {
    "firewood": 1, "charcoal": 2, "kerosene": 3,
    "LPG": 4, "electric": 5
}
df.loc[df["income_proxy"].isna(), "income_proxy"] = df.loc[df["income_proxy"].isna(), "cooking_fuel"].map(fuel_income_map)


In [8]:
from sklearn.preprocessing import StandardScaler

features = df[["consumption_kwh", "income_proxy", "appliance_index"]]
X = StandardScaler().fit_transform(features)


In [9]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# KMeans
kmeans = KMeans(n_clusters=4, random_state=42)
df["segment_kmeans"] = kmeans.fit_predict(X)

# GMM
gmm = GaussianMixture(n_components=4, random_state=42)
df["segment_gmm"] = gmm.fit_predict(X)


ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [10]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
df["income_proxy"] = imputer.fit_transform(df[["income_proxy"]])

print(df["income_proxy"].isna().sum())  # should print 0


0


In [11]:
df["income_proxy"] = pd.to_numeric(df["income_proxy"], errors="coerce")
df["income_proxy"] = imputer.fit_transform(df[["income_proxy"]])


In [12]:
features = df[["consumption_kwh", "income_proxy", "appliance_index"]]

# Drop any remaining NaNs just in case
features = features.dropna()

from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(features)


In [13]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

kmeans = KMeans(n_clusters=4, random_state=42)
df.loc[features.index, "segment_kmeans"] = kmeans.fit_predict(X)

gmm = GaussianMixture(n_components=4, random_state=42)
df.loc[features.index, "segment_gmm"] = gmm.fit_predict(X)


In [14]:
segment_map = {0:"A", 1:"B", 2:"C", 3:"D"}
df["segment_label"] = df["segment_kmeans"].map(segment_map)


In [17]:
df[["household_id","consumption_kwh","income_proxy","segment_label"]].to_csv(
    "data/processed/segmented_households.csv", index=False
)


In [16]:
import os

# Ensure the processed folder exists
os.makedirs("data/processed", exist_ok=True)

# Now save the file
df[["household_id","consumption_kwh","income_proxy","segment_label"]].to_csv(
    "data/processed/segmented_households.csv", index=False
)
